In [5]:
import pandas as pd
import joblib
from sklearn.metrics import classification_report

x_test = pd.read_parquet('../data/processed/v1/x_test_processed.parquet')
y_test = pd.read_parquet('../data/processed/v1/y_test.parquet')

model = joblib.load(f'../models/models/v1/best_model_overall_1.pkl')
probs = model.predict_proba(x_test)[:, 1]

custom_threshold = 0.85
y_pred_custom = (probs >= custom_threshold).astype(int)

print(classification_report(y_test, y_pred_custom))
print(pd.Series(y_pred_custom).describe())

              precision    recall  f1-score   support

           0       0.72      0.92      0.81     44943
           1       0.92      0.71      0.80     56099

    accuracy                           0.80    101042
   macro avg       0.82      0.82      0.80    101042
weighted avg       0.83      0.80      0.80    101042

count    101042.000000
mean          0.430168
std           0.495102
min           0.000000
25%           0.000000
50%           0.000000
75%           1.000000
max           1.000000
dtype: float64


In [6]:
import numpy as np
from sklearn.metrics import classification_report

for thresh in np.arange(0.60, 0.90, 0.05):
    preds = (probs >= thresh).astype(int)
    print(f"--- Threshold: {thresh:.2f} ---")
    print(classification_report(y_test, preds, digits=2))

--- Threshold: 0.60 ---
              precision    recall  f1-score   support

           0       0.99      0.86      0.92     44943
           1       0.90      1.00      0.94     56099

    accuracy                           0.93    101042
   macro avg       0.95      0.93      0.93    101042
weighted avg       0.94      0.93      0.93    101042

--- Threshold: 0.65 ---
              precision    recall  f1-score   support

           0       0.99      0.86      0.92     44943
           1       0.90      0.99      0.94     56099

    accuracy                           0.93    101042
   macro avg       0.94      0.93      0.93    101042
weighted avg       0.94      0.93      0.93    101042

--- Threshold: 0.70 ---
              precision    recall  f1-score   support

           0       0.99      0.86      0.92     44943
           1       0.90      0.99      0.94     56099

    accuracy                           0.93    101042
   macro avg       0.94      0.92      0.93    101042
we

In [14]:
import os
from pathlib import Path
import pandas as pd
from sklearn import set_config
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler, OrdinalEncoder, FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer, make_column_selector
import joblib

PROJECT_ROOT = Path.cwd().resolve()
sub_cat = ['Basic', 'Standard', 'Premium']
con_len_cat = ['Monthly', 'Quarterly', 'Annual']

def convert_data_type(x):
    x_copy = x.copy()
    x_copy = x_copy.convert_dtypes()

    columns = x_copy.columns.drop('remainder__num___total_spend', errors='ignore')
    
    for column in columns:
        x_copy[column] = x_copy[column].astype('int32')
    
    if 'remainder__num___total_spend' in x_copy.columns:
        x_copy['remainder__num___total_spend'] = x_copy['remainder__num___total_spend'].astype('float32')
    
    return x_copy

def col_rename(x):
    x_copy = x.copy()
    x_copy.columns = [col.lower().replace(' ', '_') for col in x_copy.columns]
    x_copy.drop(columns='customerid', errors='ignore', inplace=True)
    return x_copy

def preprocessor(data, fit_transform='yes', std='yes', ver=1):
    data_copy = data.copy()

    if fit_transform.lower().strip() == 'yes':
        cat_impute = Pipeline(steps=[
            ('cat_impute', SimpleImputer(strategy='most_frequent'))
        ])
        
        num_impute = Pipeline(steps=[
            ('num_impute', SimpleImputer(strategy='median'))
        ])
        
        impute_full = ColumnTransformer(transformers=[
            ('cat_', cat_impute, make_column_selector(dtype_exclude='number', dtype_include=['string','object'])),
            ('num_', num_impute, make_column_selector(dtype_exclude='string', dtype_include='number'))
            ],
            verbose_feature_names_out=True,
            remainder='passthrough'
        )
        
        encode = ColumnTransformer(
            transformers=[
                ('enc_', OrdinalEncoder(categories=[sub_cat, con_len_cat]), [1, 2]),
                ('enc', OneHotEncoder(drop='first', sparse_output=False), [0])
            ],
            verbose_feature_names_out=True,
            remainder='passthrough'
        )

        pipeline_normal = Pipeline(steps=[
            ('col_rename', FunctionTransformer(col_rename)),
            ('imputer', impute_full),
            ('encoder', encode),
            ('reduce_size', FunctionTransformer(convert_data_type)),
            ('feature_eng', FunctionTransformer(feature_engineering))
        ])
        
        pipeline_normal.set_output(transform="pandas")
        data_transformed = pipeline_normal.fit_transform(data_copy)
        
        pipeline_path = Path(f'../models/pipelines/v{ver}/pipeline_normal_v{ver}.joblib')
        pipeline_path.parent.mkdir(parents=True, exist_ok=True)
        joblib.dump(pipeline_normal, pipeline_path)

        if std.lower().strip() == 'yes':
            data_transformed_std = standardize(data_transformed, fit_transform='yes', ver=ver)
            return data_transformed, data_transformed_std
        return data_transformed
    else:
        pipeline_path = Path(f'../models/pipelines/v{ver}/pipeline_normal_v{ver}.joblib')
        if not pipeline_path.exists():
            raise FileNotFoundError(f"Pipeline file not found at {pipeline_path}.")
        
        pipeline_normal = joblib.load(f'../models/pipelines/v{ver}/pipeline_normal_v{ver}.joblib')
        pipeline_normal.set_output(transform="pandas")
        data_transformed = pipeline_normal.transform(data_copy)

        if std.lower().strip() == 'yes':
            data_transformed_std = standardize(data_transformed, fit_transform='no', ver=ver)
            return data_transformed, data_transformed_std
        return data_transformed

def standardize(data, fit_transform='yes', ver=1):
    data_copy = data.copy()
    all_columns = data_copy.columns.tolist()
    
    exclude_cols = [
        "enc___cat___subscription_type",
        "enc___cat___contract_length",
        "enc__cat___gender_Male",
        "is_new_customer",
        "is_long_term",
        "low_usage",
        "high_usage",
        "high_support_high_value",
        "severe_payment_delay",
        "high_spend_low_usage",
        "inactive_customer",
        "high_value_customer",
        "high_sub_type_low_usage",
    ]

    features_to_scale = [col for col in all_columns if col not in exclude_cols]
    scaler_path = Path(f'../models/scalers/v{ver}/scaler_v{ver}.joblib')
    scaler_path.parent.mkdir(parents=True, exist_ok=True)

    if fit_transform.lower().strip() == 'yes':
        scaler = ColumnTransformer(
            transformers=[('scaler', StandardScaler(), features_to_scale)],
            remainder='passthrough',
            verbose_feature_names_out=False
        )
        scaler.set_output(transform="pandas")
        data_copy_scaled = scaler.fit_transform(data_copy)
        data_copy = data_copy_scaled[all_columns]
        joblib.dump(scaler, scaler_path)
        return data_copy
    else:
        if not scaler_path.exists():
            raise FileNotFoundError(f"Scaler file not found at {scaler_path}.")
        scaler = joblib.load(scaler_path)
        scaler.set_output(transform="pandas")
        data_copy = scaler.transform(data_copy)
        return data_copy[all_columns]

def feature_engineering(x):
    x_copy = x.copy()
    x_copy["tenure_per_age"] = (x_copy["remainder__num___tenure"] / (x_copy["remainder__num___age"] + 1)).astype('float32')
    x_copy["usage_per_tenure"] = (x_copy["remainder__num___usage_frequency"] / (x_copy["remainder__num___tenure"] + 1)).astype('float32')
    x_copy["is_new_customer"] = (x_copy["remainder__num___tenure"] <= 6).astype('int8')
    x_copy["is_long_term"] = (x_copy["remainder__num___tenure"] >= 24).astype('int8')
    x_copy["low_usage"] = (x_copy["remainder__num___usage_frequency"] <= 5).astype("int8")
    x_copy["high_usage"] = (x_copy["remainder__num___usage_frequency"] >= 20).astype("int8")
    x_copy["support_calls_per_tenure"] = (x_copy["remainder__num___support_calls"] / (x_copy["remainder__num___tenure"] + 1)).astype('float32')
    x_copy["high_support_high_value"] = ((x_copy["remainder__num___support_calls"] >= 5) & (x_copy["remainder__num___total_spend"] >= 700)).astype('int8')
    x_copy["support_usage_ratio"] = (x_copy["remainder__num___support_calls"] / (x_copy["remainder__num___usage_frequency"] + 1)).astype('float32')
    x_copy["severe_payment_delay"] = (x_copy["remainder__num___payment_delay"] > 7).astype('int8')
    x_copy["spend_per_tenure"] = (x_copy["remainder__num___total_spend"] / (x_copy["remainder__num___tenure"] + 1)).astype('float32')
    
    x_copy["high_spend_low_usage"] = (
        (x_copy["remainder__num___total_spend"] > x_copy["remainder__num___total_spend"].median()) &
        (x_copy["remainder__num___usage_frequency"] < x_copy["remainder__num___usage_frequency"].median())).astype('int8')

    x_copy["inactive_customer"] = (x_copy["remainder__num___last_interaction"] >= 30).astype('int8')

    x_copy["churn_risk_signals"] = (
        (x_copy["remainder__num___last_interaction"] >= 30).astype(int)
        + (x_copy["remainder__num___usage_frequency"] <= 5).astype(int)
        + (x_copy["remainder__num___support_calls"] >= 5).astype(int)
        + (x_copy["remainder__num___payment_delay"] > 0).astype(int)).astype('int16')

    x_copy["engagement_score"] = (
        x_copy["remainder__num___usage_frequency"]
        / (x_copy["remainder__num___last_interaction"] + 1)).astype('float32')

    x_copy["customer_friction"] = (
        x_copy["remainder__num___support_calls"]
        + x_copy["remainder__num___payment_delay"]).astype('int16')
    
    high_value_threshold = (x_copy["remainder__num___total_spend"].quantile(0.75)).astype(int)
    x_copy["high_value_customer"] = (x_copy["remainder__num___total_spend"] >= high_value_threshold).astype('int8')
    x_copy["high_sub_type_low_usage"] = ((x_copy["remainder__num___total_spend"] > 800) & (x_copy["remainder__num___usage_frequency"] < 10)).astype('int8')

    x_copy.drop(columns='remainder__num___payment_delay', inplace=True, errors='ignore')
    return x_copy

def y_preprocessor(y, fit_transform='yes', ver=1):
    y_copy = pd.DataFrame(y).copy()
    y_copy.columns = y_copy.columns.str.lower()

    imputer_path = Path(f'../models/imputers/v{ver}/y_imputer_v{ver}.joblib')
    imputer_path.parent.mkdir(parents=True, exist_ok=True)

    if fit_transform.lower().strip() == 'yes':
        y_imputer = ColumnTransformer(transformers=[
            ('impute', SimpleImputer(strategy='most_frequent'), ['churn'])
        ])
        y_imputer.set_output(transform="pandas")
        y_transformed = y_imputer.fit_transform(y_copy)
        y_transformed.columns = [col.split('__')[-1] for col in y_transformed.columns]
        y_transformed = y_transformed.astype('int8')
        joblib.dump(y_imputer, imputer_path)
        return y_transformed
    else:
        if not imputer_path.exists():
            raise FileNotFoundError(f"Imputer file not found at {imputer_path}.")
        y_imputer = joblib.load(imputer_path)
        y_imputer.set_output(transform="pandas")
        y_transformed = y_imputer.transform(y_copy)
        y_transformed.columns = [col.split('__')[-1] for col in y_transformed.columns]
        y_transformed = y_transformed.astype('int8')
        return y_transformed

def load_data(ver=1, base_only='no'):
    data_1 = pd.read_csv(PROJECT_ROOT / "data/raw/customer_churn_dataset-testing-master.csv")
    data_2 = pd.read_csv(PROJECT_ROOT / "data/raw/customer_churn_dataset-training-master.csv")
    data_full = pd.concat([data_1, data_2])
    
    X, y = data_full.drop(columns='Churn'), data_full[['Churn']]
    y = y_preprocessor(y, fit_transform='yes', ver=ver)
    Xtr, Xte, yp, yt = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
    
    if base_only == 'yes':
        return pd.concat([Xtr, yp], axis=1)
    return Xtr, Xte, yp, yt

In [22]:
import sys
from pathlib import Path

# Get the absolute path to your project root (customer-churn-ml)
# Assuming your notebook is inside a folder like 'notebooks/'
project_root = Path.cwd().parent 

if str(project_root) not in sys.path:
    sys.path.append(str(project_root))
    

data_1 = pd.read_csv(f"../data/raw/customer_churn_dataset-testing-master.csv")
data_2 = pd.read_csv(f"../data/raw/customer_churn_dataset-training-master.csv")
data_full = pd.concat([data_1, data_2])

X, y = data_full.drop(columns='Churn'), data_full[['Churn']]
y = y_preprocessor(y, fit_transform='no', ver=1)
Xtr, Xte, yp, yt = train_test_split(X, y, test_size=0.00001, stratify=y, random_state=42)


df_raw = pd.DataFrame(pd.concat([Xte, yt], axis=1))
print("Raw DataFrame columns:", df_raw.columns.tolist())
print(df_raw)
x, xs = preprocessor(df_raw, "no", "yes")
print("Processed feature matrix sample:\n", x)
print(yt)

Raw DataFrame columns: ['CustomerID', 'Age', 'Gender', 'Tenure', 'Usage Frequency', 'Support Calls', 'Payment Delay', 'Subscription Type', 'Contract Length', 'Total Spend', 'Last Interaction', 'churn']
        CustomerID   Age  Gender  Tenure  Usage Frequency  Support Calls  \
338077    345192.0  38.0    Male    44.0             23.0            0.0   
42823      42824.0  40.0    Male    39.0             10.0            7.0   
56617      56618.0  34.0    Male    41.0             18.0            5.0   
362986    370101.0  42.0    Male    49.0              8.0            1.0   
348051    355166.0  31.0    Male    11.0             30.0            0.0   
4189        4198.0  18.0  Female    31.0              2.0            1.0   

        Payment Delay Subscription Type Contract Length  Total Spend  \
338077            5.0           Premium          Annual       967.50   
42823            27.0           Premium         Monthly       155.00   
56617            20.0           Premium       Qua